# AMPR Phase 3 — Train MF Branch on Kaggle 2×T4

Run on **Kaggle Notebook** with 2×T4 GPU (~2-3h).

Required Kaggle Datasets (attach before running):
- `ampr-phase3-embeddings` — ESM-2 HDF5, PPI npy files
- `ampr-pdbch` — PDBch labels, DAG matrices, splits, protein_order, GO embeddings, cmap_all.h5

Expected: `val Fmax (DAG-prop)` ≥ 0.30 after 10 epochs (vs AMPR v1 baseline 0.158).

In [ ]:
import subprocess, os
os.makedirs('/kaggle/working/datn', exist_ok=True)
!git clone https://github.com/YOUR_USERNAME/datn /kaggle/working/datn
%cd /kaggle/working/datn
!pip install -q transformers==4.41.2 obonet biopython==1.84 h5py pyyaml tqdm

In [ ]:
import os
os.makedirs('data/embeddings', exist_ok=True)
os.makedirs('data/contact_maps', exist_ok=True)
os.makedirs('data/pdbch', exist_ok=True)

# Symlink embeddings
!ln -sf /kaggle/input/ampr-phase3-embeddings/esm2_residue.h5 data/embeddings/esm2_residue.h5
!ln -sf /kaggle/input/ampr-phase3-embeddings/ppi_deepgo.npy data/embeddings/ppi_deepgo.npy
!ln -sf /kaggle/input/ampr-phase3-embeddings/ppi_deepgo_mask.npy data/embeddings/ppi_deepgo_mask.npy

# Symlink PDBch artifacts
!ln -sf /kaggle/input/ampr-pdbch/cmap_all.h5 data/contact_maps/cmap_all.h5
!ln -sf /kaggle/input/ampr-pdbch/labels_mf.npy data/pdbch/labels_mf.npy
!ln -sf /kaggle/input/ampr-pdbch/dag_matrix_mf.npy data/pdbch/dag_matrix_mf.npy
!ln -sf /kaggle/input/ampr-pdbch/splits.json data/pdbch/splits.json
!ln -sf /kaggle/input/ampr-pdbch/protein_order.json data/pdbch/protein_order.json
!ln -sf /kaggle/input/ampr-pdbch/go_emb_mf.npy data/embeddings/go_emb_mf.npy

print('Symlinks ready.')

In [ ]:
# Verify GPU setup
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# Train MF v3 — logs val Fmax (raw) and val Fmax (DAG-prop) each epoch
!python main.py --config configs/mf_v3.yaml 2>&1 | tee /kaggle/working/mf_v3_train.log

In [ ]:
# Archive checkpoint + log
import shutil
shutil.copy('checkpoints/mf_v3/best.pt', '/kaggle/working/mf_v3_best.pt')
print('Training complete. Checkpoint saved to /kaggle/working/')